This line installs the **FAISS library** (Facebook AI Similarity Search) specifically for CPU-based systems using pip.

In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 22.1 MB/s eta 0:00:00


Imports essential libraries:

*   faiss for efficient similarity search using vector embeddings.
*   pandas, numpy for data manipulation and numerical operations.
*   SentenceTransformer for converting text into embeddings.
*   pickle, json, os, time for file handling, serialization, and timing.
---

Defines key file paths:
*   csv_path: Path to the CSV file with news metadata.
*   cache_path: Path to a cache file to store already scraped articles.







In [2]:
import faiss
import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup
import time
import os
import pickle
import json

# Paths
csv_path = "/content/drive/MyDrive/files for ML/news.news_metadata.csv"
cache_path = "/content/drive/MyDrive/files for ML/cached_scraped_articles.pkl"

 # **Step 1: Define Scraping Function**

1.  Sets a user-agent header to mimic a browser request and avoid being blocked by the website.
2.   Defines scrape_cointelegraph_url(news_url):

*   Sends a GET request to the given CoinTelegraph article URL.
*   Attempts to extract the main article content from the articles.
*   Returns clean text if found, otherwise returns None.
*   Includes error handling for failed requests or timeouts.

In [3]:
# Step 1: Define scraping function
REQUEST_HEADER = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/102.0.0.0 Safari/537.36'
}

def scrape_cointelegraph_url(news_url):
    try:
        response = requests.get(news_url, headers=REQUEST_HEADER, timeout=10)
        if response.ok:
            soup = BeautifulSoup(response.content, "html.parser")
            article_tag = soup.find("article")
            return article_tag.get_text(strip=True) if article_tag else None
    except requests.exceptions.RequestException as e:
        print(f"Failed to scrape {news_url}: {e}")
    return None

# **Step 2: Load or Scrape Articles**
Checks for a cached pickle file to avoid re-scraping previously processed articles.

1.   If cache exists: loads the scraped article data directly using pickle.
2.   If not:

*   Reads the news metadata CSV.
*   Selects relevant columns: title, source_article_description, news_url, and publish_time.
*   Converts publish_time to datetime format.
*   Scrapes the full article text from each URL using the previously defined function.
*   Replaces missing text with an empty string and removes articles with too little content.
*   Saves the processed data to a cache file for future runs.

In [4]:
# Step 2: Load or scrape articles
if os.path.exists(cache_path):
    print(" Loading cached articles from pickle file...")
    with open(cache_path, "rb") as f:
        df_news = pickle.load(f)
else:
    print("Scraping articles for the first time...")
    df_news = pd.read_csv(csv_path)
    df_news = df_news[["title", "source_article_description", "news_url", "publish_time"]]
    df_news["publish_time"] = pd.to_datetime(df_news["publish_time"])
    df_news["full_text"] = df_news["news_url"].apply(scrape_cointelegraph_url)
    df_news["full_text"].fillna("", inplace=True)
    df_news = df_news[df_news["full_text"].str.len() > 50]
    with open(cache_path, "wb") as f:
        pickle.dump(df_news, f)
    print("Scraping complete. Articles cached for future use.")

 Loading cached articles from pickle file...


# **Step 3: Embedding**
Loads the MiniLM-L6-v2 model from SentenceTransformers for generating sentence embeddings.

Applies the model to each article's full text to generate vector embeddings.

Skips empty or whitespace-only articles.

Collects valid embeddings into a list and stacks them into a NumPy array for further processing.

If no valid embeddings are found, creates an empty NumPy array.

In [5]:
# Step 3: Embedding
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
df_news["embedding"] = df_news["full_text"].apply(lambda x: embedder.encode(x) if x.strip() else None)
valid_embeddings = df_news["embedding"].dropna().tolist()

if valid_embeddings:
    article_embeddings = np.vstack(valid_embeddings)
else:
    article_embeddings = np.array([])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **Step 4: FAISS Index**

1.   Checks if there are valid article embeddings to work with.
2.   If available:

*   Retrieves the embedding dimension.
*   Creates a FAISS index using IndexFlatL2 (for L2 or Euclidean distance).
*   Adds all article embeddings to the index for fast similarity search.
*   If no embeddings exist, it logs a message indicating nothing was added.

In [6]:
# Step 4: FAISS Index
if article_embeddings.size > 0:
    dimension = article_embeddings.shape[1]
    faiss_index = faiss.IndexFlatL2(dimension)
    faiss_index.add(article_embeddings)
else:
    print("No embeddings to store in FAISS.")

# **Step 5: Improved Retrieval and Ranking**


1.   rank_retrieved_articles(query, retrieved_articles):

*   Enhances FAISS results by combining multiple signals:
*   *Keyword overlap: Matches between query words and article text.*
*   *FAISS index position: Normalized to reflect similarity rank.*
*   *Time decay: Gives preference to more recent articles.*
*   Combines these into a final relevance score for ranking.
---
2.  retrieve_articles_faiss(query, top_n=3):
*   Encodes the query into an embedding and searches the FAISS index for similar articles.
*   Fetches more than needed (top_n * 2) to allow better post-filtering.
*   Filters out articles older than 6 months.
*   Returns the top N most relevant articles using the custom ranking function.

In [7]:
# Step 5: Improved Retrieval and Ranking
def rank_retrieved_articles(query, retrieved_articles):
    if retrieved_articles.empty:
        return retrieved_articles

    query_words = set(query.lower().split())
    retrieved_articles["keyword_overlap"] = retrieved_articles["full_text"].apply(
        lambda text: len(set(text.lower().split()) & query_words)
    )

    max_overlap = retrieved_articles["keyword_overlap"].max()
    retrieved_articles["keyword_score"] = retrieved_articles["keyword_overlap"] / (max_overlap + 1e-5)
    retrieved_articles["faiss_score"] = retrieved_articles.index / retrieved_articles.index.max()

    latest_date = retrieved_articles["publish_time"].max()
    retrieved_articles["time_decay"] = retrieved_articles["publish_time"].apply(
        lambda x: 1 / (1 + (latest_date - x).days)
    )

    retrieved_articles["relevance_score"] = (
        0.5 * retrieved_articles["keyword_score"] +
        0.3 * retrieved_articles["faiss_score"] +
        0.2 * retrieved_articles["time_decay"]
    )

    return retrieved_articles.sort_values(by="relevance_score", ascending=False)

def retrieve_articles_faiss(query, top_n=3):
    if article_embeddings.size == 0:
        return pd.DataFrame()

    query_embedding = embedder.encode(query).reshape(1, -1)
    _, top_indices = faiss_index.search(query_embedding, top_n * 2)
    retrieved_articles = df_news.iloc[top_indices[0]].copy()

    six_months_ago = pd.Timestamp.now(tz="UTC") - pd.DateOffset(months=6)
    retrieved_articles = retrieved_articles[retrieved_articles["publish_time"] >= six_months_ago]

    return rank_retrieved_articles(query, retrieved_articles).head(top_n)

# **Step 6: Together.AI API Call**

1.   Defines the API key and function to interact with the Mistral-7B-Instruct model hosted on Together.AI.
2.   query_mistral7b(prompt, retries=3, wait_time=5):

*   Sends a POST request with the prompt and generation parameters (max_tokens, temperature, top_p).
*   Retries the request up to 3 times in case of failure, with a short delay between attempts.
*   Returns the generated response text if successful, or an error message otherwise.

In [8]:
# Step 6: Together.AI Call
TOGETHER_AI_API_KEY = "ypur_api_key"

def query_mistral7b(prompt, retries=3, wait_time=5):
    url = "https://api.together.xyz/v1/completions"
    headers = {
        "Authorization": f"Bearer {TOGETHER_AI_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "mistralai/Mistral-7B-Instruct-v0.1",
        "prompt": prompt,
        "max_tokens": 150,
        "temperature": 0.7,
        "top_p": 0.9
    }

    for _ in range(retries):
        response = requests.post(url, headers=headers, json=payload)
        try:
            response_json = response.json()
            if response.status_code == 200 and "choices" in response_json:
                return response_json["choices"][0].get("text", "No response received.")
        except Exception as e:
            print(f"Error: {e}")
        time.sleep(wait_time)
    return "API is unavailable after multiple retries."

# **Step 7: Response Formatter**
1.   Defines a function to structure the final output in a clean and readable format.
2.   format_response(response_text, retrieved_articles):


*   Returns a dictionary containing:
*   The generated text response from the language model.

A list of retrieved articles, each with:

Title

URL

Publication date

A short snippet (first 300 characters) of the article text

In [9]:
# Step 7: Response Formatter
def format_response(response_text, retrieved_articles):
    return {
        "generated_response": response_text.strip(),
        "retrieved_articles": [
            {
                "title": row["title"],
                "url": row["news_url"],
                "publish_time": row["publish_time"].strftime("%Y-%m-%d"),
                "summary": row["full_text"][:300] + "..."
            }
            for _, row in retrieved_articles.iterrows()
        ]
    }

# **Step 8: RAG (Retrieval-Augmented Generation) Execution**
generate_response(query) combines all the previous steps into one unified pipeline:

Retrieves relevant articles using FAISS.

If no articles are found, returns an error message.

Otherwise, constructs a context by concatenating article texts.

Sends the context + query as a prompt to the Mistral-7B model.

Returns a formatted response combining the generated answer and article metadata.

In [10]:
# Step 8: RAG Execution
def generate_response(query):
    retrieved_articles = retrieve_articles_faiss(query)
    if retrieved_articles.empty:
        return {"error": "No relevant articles found."}
    context = "\n\n".join(retrieved_articles["full_text"].tolist())
    response = query_mistral7b(f"Based on the following news:\n{context}\n\nAnswer this query: {query}")
    return format_response(response, retrieved_articles)

# **Step 9: Run**
Sets a sample user query: "What is the trend for bitcoin in 2024?"

Calls the generate_response function to:

Retrieve relevant articles,

Generate a response using the language model,

Format the result.

Prints the final output as nicely formatted JSON, showing both the generated response and the related articles.

In [12]:
# Step 9: Run
query = "What is the trend for bitcoin in 2024?"
response = generate_response(query)
print(json.dumps(response, indent=4))

{
    "generated_response": "The trend for Bitcoin in 2024 was that it solidified its position in global finance and hit the milestone price mark of $100,000. Bitcoin's role in global finance continued to expand, and discussions about its potential as a global reserve asset moved from niche speculation into the mainstream.",
    "retrieved_articles": [
        {
            "title": "Crypto industry report 2025: Key trends, insights and growth opportunities",
            "url": "https://cointelegraph.com/news/crypto-industry-report-2025-key-trends-insights-and-growth-opportunities",
            "publish_time": "2025-01-29",
            "summary": "Nick MJan 29, 2025Crypto industry report 2025: Key trends, insights and growth opportunitiesCointelegraph Research delves into 2024\u2019s defining trends, analyzing Bitcoin\u2019s historic rise, DeFi\u2019s recovery, altcoin dynamics and regulatory shifts.11095Total views21Total sharesListen to article0:00ReportT..."
        }
    ]
}
